# 섹션3-11. AI를 활용한 아날로그 센서 데이터의 Smoothing - 이동평균선, Peak 포락선, Savitzky-Golay Filter

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 22강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

> 강의 제공 센서 데이터는 `.gitignore` 처리했으니(`references/lecture_data/sensor_smoothing_studio/`),
> 강의가 설명한 특징(삐끗삐끗한 피크, 위아래로 증가, 진동처럼 울퉁불퉁)을 그대로 갖춘 합성
> 신호를 만들어 재현한다. **다만 이번엔 "진짜 신호"를 알고 만들기 때문에, 어떤 스무딩이
> 실제로 더 정확한지 눈이 아니라 숫자로 잴 수 있다** — 강의에서는 "제 눈에는 피크 포락선이
> 가장 보기 좋다"로 끝났는데, 그 판단이 맞는지 검증해본다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

from scipy.signal import find_peaks, savgol_filter

n = 600
t = np.arange(n)
true_signal = 5 * np.sin(2 * np.pi * t / 150) + 0.01 * t  # 우리만 아는 "정답" 신호
noise = rng.normal(0, 0.6, n)
spike_idx = rng.choice(n, 25, replace=False)
spikes = np.zeros(n)
spikes[spike_idx] = rng.normal(0, 4, 25)

raw = true_signal + noise + spikes
print(f"표본 {n}개, 스파이크 {len(spike_idx)}개")

### 1. 원본 — 강의가 설명한 그대로 삐죽삐죽하다

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t, raw, lw=0.8, color="#4C78A8")
ax.set_title("원본 센서 신호 — 노이즈 + 스파이크 25개")
plt.show()

### 2. 세 가지 스무딩 — 강의가 다룬 방법 그대로

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


# ① 이동평균
moving_avg = np.convolve(raw, np.ones(21) / 21, mode="same")

# ② 피크 포락선 — 국소 최댓값/최솟값을 이어서 상단·하단 포락선을 만들고 평균낸다
peaks, _ = find_peaks(raw, distance=15)
troughs, _ = find_peaks(-raw, distance=15)
upper = np.interp(t, peaks, raw[peaks])
lower = np.interp(t, troughs, raw[troughs])
envelope = (upper + lower) / 2

# ③ Savitzky-Golay 필터 — 구간마다 다항식을 적합해 스무딩
savgol = savgol_filter(raw, window_length=21, polyorder=3)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, smoothed, name in zip(axes, [moving_avg, envelope, savgol],
                               ["이동평균", "피크 포락선", "Savitzky-Golay"]):
    ax.plot(t, raw, lw=0.5, alpha=0.35, color="gray", label="원본")
    ax.plot(t, smoothed, lw=1.8, color="#E45756", label=name)
    ax.set_title(f"{name}  (RMSE vs 진짜신호 = {rmse(smoothed, true_signal):.3f})")
    ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

print(f"원본 자체의 오차(원본-진짜신호) RMSE = {rmse(raw, true_signal):.3f}  ← 스무딩 안 한 기준선")

> **피크 포락선의 RMSE가 원본(스무딩 전)보다 크다** — 스무딩을 했는데 오히려 더 나빠졌다.
> 이유는 포락선이 국소 최댓값·최솟값을 **그대로 통과**하기 때문이다. 스파이크가 우연히
> "국소 최댓값"으로 잡히면 그 값이 그대로 포락선에 꿰어져 들어간다. 이동평균과
> Savitzky-Golay는 구간을 평균/적합하기 때문에 스파이크 하나가 희석된다.

### 3. 스파이크가 없었다면? — 같은 방법, 다른 결과

In [ ]:
raw_clean = true_signal + noise  # 스파이크 없이 노이즈만

ma_c = np.convolve(raw_clean, np.ones(21) / 21, mode="same")
pk, _ = find_peaks(raw_clean, distance=15)
tr, _ = find_peaks(-raw_clean, distance=15)
env_c = (np.interp(t, pk, raw_clean[pk]) + np.interp(t, tr, raw_clean[tr])) / 2
sg_c = savgol_filter(raw_clean, window_length=21, polyorder=3)

print("             스파이크 있음   스파이크 없음")
print(f"이동평균      {rmse(moving_avg, true_signal):>10.3f}   {rmse(ma_c, true_signal):>10.3f}")
print(f"피크 포락선   {rmse(envelope, true_signal):>10.3f}   {rmse(env_c, true_signal):>10.3f}")
print(f"Savitzky-Golay{rmse(savgol, true_signal):>10.3f}   {rmse(sg_c, true_signal):>10.3f}")

**정리.** 스파이크가 없으면 피크 포락선도 이동평균만큼 준수하다(0.33 근처). 스파이크가
섞이면 이동평균·Savitzky-Golay는 거의 그대로인데 **피크 포락선만 크게 나빠진다.**

| | 스파이크 없음 | 스파이크 있음 | 스파이크에 강한가 |
| --- | --- | --- | --- |
| 이동평균 | 좋음 | 좋음 | ✅ |
| 피크 포락선 | 좋음 | **원본보다 나쁨** | ❌ |
| Savitzky-Golay | 가장 좋음 | 가장 좋음 | ✅ |

강의는 "이동평균·피크포락선·골레이필터 중 제 눈에는 피크포락선이 가장 보기 좋다"고
마무리했는데, **눈으로 매끄러워 보이는 것과 실제로 정확한 것은 다르다** — 특히 데이터에
이상치성 스파이크가 섞여 있다면 피크 포락선은 오히려 위험한 선택이 될 수 있다.
센서 데이터를 다룰 때는 "스파이크가 이상치인가, 진짜 신호인가"부터 먼저 판단해야 한다.


---

## 메모

-
